|<h2>Course:</h2>|<h1><a href="https://udemy.com/course/deeplearning_x/?couponCode=202508" target="_blank">A deep understanding of deep learning</a></h1>|
|-|:-:|
|<h2>Section:</h2>|<h1>RNNs (and LSTM and GRU)<h1>|
|<h2>Lecture:</h2>|<h1><b>The RNN class<b></h1>|

<br>

<h5><b>Teacher:</b> Mike X Cohen, <a href="https://sincxpress.com" target="_blank">sincxpress.com</a></h5>
<h5><b>Course URL:</b> <a href="https://udemy.com/course/deeplearning_x/?couponCode=202508" target="_blank">udemy.com/course/deeplearning_x/?couponCode=202508</a></h5>
<i>Using the code without the course may lead to confusion or errors.</i>

In [1]:
### import libraries
import torch
import torch.nn as nn
import numpy as np

# Explore the RNN type

In [11]:
# set layer parameters
input_size  =  9 # number of features to extract (e.g., number of data channels)
hidden_size = 16 # number of units in the hidden state
num_layers  =  3 # number of vertical stacks of hidden layers (note: only the final layer gives an output)
actfun      = 'tanh'
bias        = True

# create an RNN instance
rnn = nn.RNN(input_size,hidden_size,num_layers,nonlinearity=actfun,bias=bias)
print(rnn)

RNN(9, 16, num_layers=3)


In [12]:
# check out the source code for more detailed info about this class
# ??nn.RNN

In [13]:
# set data parameters
seqlength = 5
batchsize = 2

# create some data
X = torch.rand(seqlength,batchsize,input_size)

# create a hidden layer (typically initialized as zeros)
hidden = torch.zeros(num_layers,batchsize,hidden_size)


# run some data through the model and show the output sizes
y,h = rnn(X,hidden)
print(f' Input shape: {list(X.shape)}')
print(f'Hidden shape: {list(h.shape)}')
print(f'Output shape: {list(y.shape)}')

 Input shape: [5, 2, 9]
Hidden shape: [3, 2, 16]
Output shape: [5, 2, 16]


In [14]:
## Default hidden state is all zeros if nothing specified:
y,h1 = rnn(X,hidden)
print(h1), print('\n\n')

y,h2 = rnn(X)
print(h2), print('\n\n')

# they're the same! (meaning default=zeros)
print(h1-h2)

tensor([[[ 0.1590, -0.1470,  0.0678,  0.7115, -0.0247,  0.4679, -0.5608,
           0.1636, -0.2523, -0.1656,  0.0193, -0.6321, -0.1588, -0.2832,
           0.4504,  0.1039],
         [ 0.2292, -0.0624, -0.2404,  0.5973, -0.0586,  0.3811, -0.3014,
           0.2822, -0.1673, -0.4979, -0.1836, -0.3888, -0.1409, -0.3260,
           0.2685,  0.0171]],

        [[-0.2465,  0.0088, -0.1537,  0.6055, -0.2251,  0.4157,  0.6353,
           0.4924,  0.4043,  0.1809,  0.2732, -0.3465,  0.0323, -0.1336,
          -0.1862, -0.3482],
         [-0.2319,  0.0772, -0.1179,  0.5985, -0.2772,  0.3822,  0.4681,
           0.4348,  0.4244,  0.3190,  0.1558, -0.0896, -0.0967, -0.1442,
          -0.1835, -0.0378]],

        [[-0.4749,  0.3624, -0.3747,  0.3706,  0.1160,  0.4084, -0.1787,
           0.1578, -0.1516, -0.4786, -0.2435, -0.0445,  0.4659,  0.4314,
          -0.1438, -0.3238],
         [-0.4560,  0.3244, -0.3314,  0.3916,  0.0372,  0.4730, -0.1885,
           0.1580, -0.1696, -0.3934, -0.1811,  0

In [15]:
# Check out the learned parameters and their sizes
for p in rnn.named_parameters():
  if 'weight' in p[0]:
    print(f'{p[0]} has size {list(p[1].shape)}')

weight_ih_l0 has size [16, 9]
weight_hh_l0 has size [16, 16]
weight_ih_l1 has size [16, 16]
weight_hh_l1 has size [16, 16]
weight_ih_l2 has size [16, 16]
weight_hh_l2 has size [16, 16]


# Create a DL model class

In [16]:
class RNNnet(nn.Module):
  def __init__(self,input_size,num_hidden,num_layers):
    super().__init__()

    # store parameters
    self.input_size = input_size
    self.num_hidden = num_hidden
    self.num_layers = num_layers

    # RNN Layer
    self.rnn = nn.RNN(input_size,num_hidden,num_layers)

    # linear layer for output
    self.out = nn.Linear(num_hidden,1)

  def forward(self,x):

    print(f'Input: {list(x.shape)}')

    # initialize hidden state for first input
    hidden = torch.zeros(self.num_layers,batchsize,self.num_hidden)
    print(f'Hidden: {list(hidden.shape)}')

    # run through the RNN layer
    y,hidden = self.rnn(x,hidden)
    print(f'RNN-out: {list(y.shape)}')
    print(f'RNN-hidden: {list(hidden.shape)}')

    # pass the RNN output through the linear output layer
    o = self.out(y)
    print(f'Output: {list(o.shape)}')

    return o,hidden

In [17]:
# create an instance of the model and inspect
net = RNNnet(input_size,hidden_size,num_layers)
print(net), print(' ')

# and check out all learnable parameters
for p in net.named_parameters():
  print(f'{p[0]} has size {list(p[1].shape)}')

RNNnet(
  (rnn): RNN(9, 16, num_layers=3)
  (out): Linear(in_features=16, out_features=1, bias=True)
)
 
rnn.weight_ih_l0 has size [16, 9]
rnn.weight_hh_l0 has size [16, 16]
rnn.bias_ih_l0 has size [16]
rnn.bias_hh_l0 has size [16]
rnn.weight_ih_l1 has size [16, 16]
rnn.weight_hh_l1 has size [16, 16]
rnn.bias_ih_l1 has size [16]
rnn.bias_hh_l1 has size [16]
rnn.weight_ih_l2 has size [16, 16]
rnn.weight_hh_l2 has size [16, 16]
rnn.bias_ih_l2 has size [16]
rnn.bias_hh_l2 has size [16]
out.weight has size [1, 16]
out.bias has size [1]


In [18]:
# test the model with some data
# create some data
X = torch.rand(seqlength,batchsize,input_size)
y = torch.rand(seqlength,batchsize,1)
yHat,h = net(X)

# try a loss function
lossfun = nn.MSELoss()
lossfun(yHat,y)

Input: [5, 2, 9]
Hidden: [3, 2, 16]
RNN-out: [5, 2, 16]
RNN-hidden: [3, 2, 16]
Output: [5, 2, 1]


tensor(0.5253, grad_fn=<MseLossBackward0>)

# Additional explorations

In [ ]:
# 1) In the video, I asked about the "l0" from the parameter name "weight_ih_l0". To explore this further,
#    recreate that RNN instance but set the number of layers to 3. Then go through the code again to print
#    out all of the weights matrices. Refer back to the discussion of layers in the previous video. Do you
#    understand the naming system of the weights matrices?
# lN - represents Nth RNN hidden weights matrix layer.
